In [ ]:
import pandas as pd
import numpy as np

# Read all CSV files
print("Reading CSV files...")
hdi_df = pd.read_csv('HDI_long.csv')
co2_df = pd.read_csv('CO2_long.csv')
disasters_df = pd.read_csv('disasters_long.csv')
i1_df = pd.read_csv('i1_long.csv')
i2_df = pd.read_csv('i2_long.csv')

# Strip whitespace from all column names
for df in [hdi_df, co2_df, disasters_df, i1_df, i2_df]:
    df.columns = df.columns.str.strip()

print(f"HDI records: {len(hdi_df)}")
print(f"CO2 records: {len(co2_df)}")
print(f"Disasters records: {len(disasters_df)}")
print(f"i1 records: {len(i1_df)}")
print(f"i2 records: {len(i2_df)}")


print("\nProcessing disasters - aggregating to country-year...")


numeric_agg = {}
for col in disasters_df.select_dtypes(include=[np.number]).columns:
    if col not in ['Year']:
        numeric_agg[col] = 'sum'

# Text/categorical columns - keep all unique values separated by " | "
text_cols = disasters_df.select_dtypes(include=['object']).columns
text_cols = [col for col in text_cols if col not in ['Country', 'ISO']]

text_agg = {}
for col in text_cols:
    text_agg[col] = lambda x: ' | '.join(sorted(set(str(v) for v in x.dropna())))

# Combine aggregations
all_agg = {**numeric_agg, **text_agg}

# Aggregate disasters
disasters_agg = disasters_df.groupby(['Country', 'ISO', 'Year']).agg(all_agg).reset_index()

# Add disaster count
disasters_agg['Disaster_Count'] = disasters_df.groupby(['Country', 'ISO', 'Year']).size().values

print(f"Aggregated disasters to {len(disasters_agg)} country-year records")

# CREATE SEPARATE COLUMNS FOR EACH DISASTER TYPE for Tableau filtering
print("\nCreating disaster type breakdowns for Tableau filters...")

# Get unique disaster types
if 'Disaster Type' in disasters_df.columns:
    unique_types = disasters_df['Disaster Type'].dropna().unique()
    print(f"Found {len(unique_types)} unique disaster types")

    # Create count columns for each disaster type
    for dtype in unique_types:
        col_name = f"Count_{dtype}".replace(' ', '_').replace('(', '').replace(')', '')
        disasters_agg[col_name] = 0

    # Fill in the counts
    type_counts = disasters_df.groupby(['Country', 'ISO', 'Year', 'Disaster Type']).size().reset_index(name='count')
    for _, row in type_counts.iterrows():
        mask = (disasters_agg['Country'] == row['Country']) & \
               (disasters_agg['ISO'] == row['ISO']) & \
               (disasters_agg['Year'] == row['Year'])
        col_name = f"Count_{row['Disaster Type']}".replace(' ', '_').replace('(', '').replace(')', '')
        if col_name in disasters_agg.columns:
            disasters_agg.loc[mask, col_name] = row['count']

# Get unique disaster groups
if 'Disaster Group' in disasters_df.columns:
    unique_groups = disasters_df['Disaster Group'].dropna().unique()
    print(f"Found {len(unique_groups)} unique disaster groups")

    for dgroup in unique_groups:
        col_name = f"Count_Group_{dgroup}".replace(' ', '_')
        disasters_agg[col_name] = 0

    group_counts = disasters_df.groupby(['Country', 'ISO', 'Year', 'Disaster Group']).size().reset_index(name='count')
    for _, row in group_counts.iterrows():
        mask = (disasters_agg['Country'] == row['Country']) & \
               (disasters_agg['ISO'] == row['ISO']) & \
               (disasters_agg['Year'] == row['Year'])
        col_name = f"Count_Group_{row['Disaster Group']}".replace(' ', '_')
        if col_name in disasters_agg.columns:
            disasters_agg.loc[mask, col_name] = row['count']

# Get unique disaster subtypes
if 'Disaster Subtype' in disasters_df.columns:
    unique_subtypes = disasters_df['Disaster Subtype'].dropna().unique()
    print(f"Found {len(unique_subtypes)} unique disaster subtypes")

    for dsubtype in unique_subtypes:
        col_name = f"Count_Subtype_{dsubtype}".replace(' ', '_').replace('(', '').replace(')', '')
        disasters_agg[col_name] = 0

    subtype_counts = disasters_df.groupby(['Country', 'ISO', 'Year', 'Disaster Subtype']).size().reset_index(name='count')
    for _, row in subtype_counts.iterrows():
        mask = (disasters_agg['Country'] == row['Country']) & \
               (disasters_agg['ISO'] == row['ISO']) & \
               (disasters_agg['Year'] == row['Year'])
        col_name = f"Count_Subtype_{row['Disaster Subtype']}".replace(' ', '_').replace('(', '').replace(')', '')
        if col_name in disasters_agg.columns:
            disasters_agg.loc[mask, col_name] = row['count']

print(f"Disasters columns after adding type breakdowns: {len(disasters_agg.columns)}")

# Process i1: Pivot to have one row per country-year
print("\nProcessing i1 - pivoting indicators to columns...")
i1_processed = i1_df.copy()

# Create unique column names combining Series Code and Series Name
i1_processed['Indicator'] = i1_processed['Series Code'].fillna('') + '__' + i1_processed['Series Name'].fillna('')

i1_pivot = i1_processed.pivot_table(
    index=['Country', 'ISO', 'Year'],
    columns='Indicator',
    values='Value1',
    aggfunc='first'
).reset_index()

# Flatten column names
i1_pivot.columns.name = None
print(f"i1 pivoted to {len(i1_pivot)} records with {len(i1_pivot.columns)} columns")

# Process i2: Pivot to have one row per country-year
print("Processing i2 - pivoting indicators to columns...")
i2_pivot = i2_df.pivot_table(
    index=['Country', 'ISO', 'Year'],
    columns='Series Name',
    values='Value2',
    aggfunc='first'
).reset_index()

i2_pivot.columns.name = None
print(f"i2 pivoted to {len(i2_pivot)} records with {len(i2_pivot.columns)} columns")

# Now join everything on ISO and Year
print("\n" + "="*60)
print("JOINING ALL DATA...")
print("="*60)

# Start with HDI
merged = hdi_df.copy()
print(f"Base (HDI): {len(merged)} rows, {len(merged.columns)} columns")

# Join CO2
merged = merged.merge(
    co2_df,
    on=['ISO', 'Year'],
    how='left',
    suffixes=('', '_co2_dup')
)
# Drop duplicate Country column
if 'Country_co2_dup' in merged.columns:
    merged.drop('Country_co2_dup', axis=1, inplace=True)
print(f"After CO2: {len(merged)} rows, {len(merged.columns)} columns")

# Join aggregated disasters
merged = merged.merge(
    disasters_agg,
    on=['ISO', 'Year'],
    how='left',
    suffixes=('', '_disaster_dup')
)
if 'Country_disaster_dup' in merged.columns:
    merged.drop('Country_disaster_dup', axis=1, inplace=True)
print(f"After Disasters: {len(merged)} rows, {len(merged.columns)} columns")

# Join i1 pivot
merged = merged.merge(
    i1_pivot,
    on=['ISO', 'Year'],
    how='left',
    suffixes=('', '_i1_dup')
)
if 'Country_i1_dup' in merged.columns:
    merged.drop('Country_i1_dup', axis=1, inplace=True)
print(f"After i1: {len(merged)} rows, {len(merged.columns)} columns")

# Join i2 pivot
merged = merged.merge(
    i2_pivot,
    on=['ISO', 'Year'],
    how='left',
    suffixes=('', '_i2_dup')
)
if 'Country_i2_dup' in merged.columns:
    merged.drop('Country_i2_dup', axis=1, inplace=True)
print(f"After i2: {len(merged)} rows, {len(merged.columns)} columns")

# Fill disaster count NaN with 0
if 'Disaster_Count' in merged.columns:
    merged['Disaster_Count'] = merged['Disaster_Count'].fillna(0)

print("\n" + "="*60)
print("FINAL RESULT")
print("="*60)
print(f"Total records: {len(merged)}")
print(f"Total columns: {len(merged.columns)}")
print(f"\nAll {len(merged.columns)} columns:")
for i, col in enumerate(merged.columns, 1):
    print(f"  {i:3d}. {col}")

# Save to CSV
output_file = 'integrated_disaster_resilience.csv'
merged.to_csv(output_file, index=False)
print(f"\n✓ Successfully saved to: {output_file}")

# Show sample
print("\n" + "="*60)
print("SAMPLE DATA (first 3 rows, first 15 columns)")
print("="*60)
print(merged.iloc[:3, :15])

print("\n" + "="*60)
print("DATA SUMMARY")
print("="*60)
print(f"Countries: {merged['Country'].nunique()}")
print(f"Years range: {merged['Year'].min()} to {merged['Year'].max()}")
print(f"Records with disasters: {(merged['Disaster_Count'] > 0).sum()}")
print(f"Records without disasters: {(merged['Disaster_Count'] == 0).sum()}")

Reading CSV files...
HDI records: 1351
CO2 records: 4608
Disasters records: 16546
i1 records: 98928
i2 records: 84096

Processing disasters - aggregating to country-year...
Aggregated disasters to 3560 country-year records

Creating disaster type breakdowns for Tableau filters...
Found 31 unique disaster types
Found 2 unique disaster groups
Found 63 unique disaster subtypes


/tmp/ipython-input-1080930447.py:98: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  disasters_agg[col_name] = 0
/tmp/ipython-input-1080930447.py:98: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  disasters_agg[col_name] = 0
/tmp/ipython-input-1080930447.py:98: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()

Disasters columns after adding type breakdowns: 116

Processing i1 - pivoting indicators to columns...
i1 pivoted to 4944 records with 23 columns
Processing i2 - pivoting indicators to columns...
i2 pivoted to 4944 records with 20 columns

JOINING ALL DATA...
Base (HDI): 1351 rows, 4 columns
After CO2: 1351 rows, 5 columns
After Disasters: 1351 rows, 118 columns
After i1: 1351 rows, 138 columns
After i2: 1351 rows, 155 columns

FINAL RESULT
Total records: 1351
Total columns: 155

All 155 columns:
    1. Country
    2. ISO
    3. Year
    4. HDI
    5. CO2
    6. End Month
    7. Start Year
    8. CPI
    9. No. Affected
   10. No. Homeless
   11. No. Injured
   12. Response Indicator
   13. Total Affected
   14. Total Damage ('000 US$)
   15. Total Deaths
   16. Disaster Group
   17. Disaster Subgroup
   18. Disaster Subtype
   19. Disaster Type
   20. Region
   21. Subregion
   22. Disaster_Count
   23. Count_Epidemic
   24. Count_Collapse_Miscellaneous
   25. Count_Drought
   26. Cou